In [1]:
from dotenv import load_dotenv
load_dotenv('../env')

from langgraph.graph import StateGraph, END
from typing import TypedDict

# Langgraph has 3 building blocks
# STATE - a dictionary that flows through the graph
#           every node reads from it and writes back from it
# NODES - python functions that take state and return updated state
# EDGES - connections between nodes, can be conditional

# Step 1 - Define the state
# TypedDict tells Langgraph exactly what keys the state contains

class SimpleState(TypedDict):
    number : int
    history : list[str]
    result : str

# Step 2 - Define nodes 
def double_it(state: SimpleState) -> SimpleState:
    """Node 1 - doubles the number"""
    new_number = state["number"] * 2
    return {
        "number" : new_number,
        "history" : state["history"] + [f"doubled to {new_number}"],
        "result" : state["result"]
    }

def add_ten(state: SimpleState) -> SimpleState:
    """Node 2 — adds 10."""
    new_number = state["number"] + 10
    return {
        "number":  new_number,
        "history": state["history"] + [f"added 10 to get {new_number}"],
        "result":  state["result"]
    }

def summarise(state: SimpleState) -> SimpleState:
    """Node 3 — creates a summary."""
    return {
        "number":  state["number"],
        "history": state["history"],
        "result":  f"Final number: {state['number']}. Steps: {' → '.join(state['history'])}"
    }

# Step 3 - Build the graph
graph = StateGraph(SimpleState)

# Add nodes
graph.add_node("double", double_it)
graph.add_node("add_ten", add_ten)
graph.add_node("summarise", summarise)

# Add edges
graph.set_entry_point("double") # start here
graph.add_edge("double", "add_ten")
graph.add_edge("add_ten", "summarise")
graph.add_edge("summarise", END)

# Compile the graph
app = graph.compile()

# Step 4 - Run it
result = app.invoke({
    "number" : 5, 
    "history" : [],
    "result": ""
})

print(f"Result: {result['result']}")
print(f"History: {result['history']}")
print(f"Final number: {result['number']}")
print()
print("State flowed through: double → add_ten → summarise → END")
print("Each node read the state, modified it, passed it forward.")


Result: Final number: 20. Steps: doubled to 10 → added 10 to get 20
History: ['doubled to 10', 'added 10 to get 20']
Final number: 20

State flowed through: double → add_ten → summarise → END
Each node read the state, modified it, passed it forward.


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class GradeState(TypedDict):
    question:    str
    answer:      str
    score:       int
    attempts:    int
    final:       str

# ── Nodes ──────────────────────────────────────────────────────────────────
def generate_answer(state: GradeState) -> dict:
    attempts = state["attempts"] + 1
    scores   = [3, 7, 9]
    score    = scores[min(attempts - 1, len(scores) - 1)]
    answer   = f"Answer attempt {attempts} (quality score: {score}/10)"
    print(f"  → Generated: {answer}")
    return {"answer": answer, "score": score, "attempts": attempts}

def improve_question(state: GradeState) -> dict:
    better_question = f"{state['question']} [more specific version]"
    print(f"  → Rewrote question: {better_question}")
    return {"question": better_question}

def finalise(state: GradeState) -> dict:
    print(f"  → Accepted answer with score {state['score']}/10")
    return {"final": state["answer"]}

# ── Conditional edge function ──────────────────────────────────────────────
def grade_answer(state: GradeState) -> str:
    if state["score"] >= 7:
        print(f"  → Score {state['score']} >= 7. Accepting.")
        return "accept"
    elif state["attempts"] < 3:
        print(f"  → Score {state['score']} < 7. Retrying (attempt {state['attempts']}).")
        return "retry"
    else:
        print(f"  → Max attempts reached. Accepting anyway.")
        return "accept"

# ── Build graph ────────────────────────────────────────────────────────────
graph = StateGraph(GradeState)

graph.add_node("generate",         generate_answer)
graph.add_node("improve_question", improve_question)
graph.add_node("finalise",         finalise)

graph.set_entry_point("generate")

graph.add_conditional_edges(
    "generate",
    grade_answer,
    {
        "accept": "finalise",
        "retry":  "improve_question"
    }
)
graph.add_edge("improve_question", "generate")
graph.add_edge("finalise", END)

app = graph.compile()

print("Running graph with conditional routing:\n")
result = app.invoke({
    "question": "What is machine learning?",
    "answer":   "",
    "score":    0,
    "attempts": 0,
    "final":    ""
})

print(f"\nFinal answer: {result['final']}")
print(f"Total attempts: {result['attempts']}")

Running graph with conditional routing:

  → Generated: Answer attempt 1 (quality score: 3/10)
  → Score 3 < 7. Retrying (attempt 1).
  → Rewrote question: What is machine learning? [more specific version]
  → Generated: Answer attempt 2 (quality score: 7/10)
  → Score 7 >= 7. Accepting.
  → Accepted answer with score 7/10

Final answer: Answer attempt 2 (quality score: 7/10)
Total attempts: 2


In [1]:
import os
from dotenv import load_dotenv
load_dotenv('../env')

import sys
sys.path.append('..')

from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable
from typing import TypedDict
from src.rag import RAGPipeline
import numpy as np

# Load pipeline
rag = RAGPipeline()
rag.load_pdf("../data/Fundamentals of Data Engineering.pdf")
print(f"Loaded - {len(rag.chunks)} chunks")

llm = ChatOllama(model="llama3.2")
parser = StrOutputParser()

# State Definition
class RAGState(TypedDict):
    question : str
    retrieved_chunks : list[str]
    distances : list[float]
    relevance_scores: list[str]
    rewritten_question: str
    context : str
    answer: str
    attempts: int
    decision: str

# Node 1 : Retrieve
def retrieve(state: RAGState) -> dict:
    """Retrieve top K chunks using current question"""
    question = state.get("rewritten_question") or state["question"]
    query_emb = rag.embedding_model.encode([question]).astype(np.float32)
    dists, idxs = rag.index.search(query_emb, 3)
    chunks = [rag.chunks[i] for i in idxs[0]]
    distances = [round(float(d), 4) for d in dists[0]]
    print(f" [retrieve] Got {len(chunks)} chunks for : '{question[:60]}'")
    return {
        "retrieved_chunks" : chunks, 
        "distances": distances, 
        "relevance_scores": [],
    }

# Node 2 : Grade chunks
grade_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a relevance grader. Given a question and a document chunk,
decide if the chunk is relevant to answering the question.

Respond with ONLY one word: 'relevant' or 'irrelevant'"""),
    ("human", "Question: {question}\n\nChunk: {chunk}\n\nIs this chunk relevant?")
])

grade_chain = grade_prompt | llm | parser

def grade_chunks(state: RAGState) -> dict:
    """Grade each retrieved chunk for relevance"""
    question =state.get("rewritten_question") or state["question"]
    scores = []
    for chunk in state["retrieved_chunks"]:
        score = grade_chain.invoke({
            "question": question, 
            "chunk": chunk[:300]
        }).strip().lower()
        score = "relevant" if "relevant" in score else "irrelevant"
        scores.append(score)
        print(f" [grade] {score} - {chunk[:60]}...")

    relevant_count = scores.count("relevant")
    decision= "generate" if relevant_count >= 1 else "rewrite"
    print (f" [grade] {relevant_count}/3 relevant -> decision: {decision}")
    return {"relevance_scores" : scores, "decision": decision}

# Node 3 : Rewrite questions
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a query rewriter. Your job is to rewrite a question 
to be more specific and likely to retrieve relevant document chunks.
Return ONLY the rewritten question, nothing else."""),
    ("human", "Original question: {question}\n\nRewrite it to be more specific:")
])

rewrite_chain = rewrite_prompt | llm | parser

def rewrite_question(state: RAGState) -> dict:
    """Rewrite the question to improve retrieval"""
    rewritten = rewrite_chain.invoke({"question": state["question"]}).strip()
    print(f"  [rewrite] '{state['question']}' → '{rewritten}'")
    return {"rewritten_question": rewritten, "attempts": state["attempts"] + 1}

# ── Node 4: Generate answer ────────────────────────────────────────────────
generate_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are DocMind, a helpful document analyst.
Answer based strictly on the provided context.
If not found, say 'I don't find this in the document.'

Context:
{context}"""),
    ("human", "{question}")
])
generate_chain = generate_prompt | llm | parser

def generate_answer(state: RAGState) -> dict:
    """Generate answer from relevant chunks."""
    question = state.get("rewritten_question") or state["question"]

    # Only use relevant chunks if we have grades
    if state["relevance_scores"]:
        good_chunks = [
            chunk for chunk, score
            in zip(state["retrieved_chunks"], state["relevance_scores"])
            if score == "relevant"
        ]
        chunks = good_chunks if good_chunks else state["retrieved_chunks"]
    else:
        chunks = state["retrieved_chunks"]

    context = "\n\n---\n\n".join(chunks)
    answer  = generate_chain.invoke({
        "context":  context,
        "question": question
    })
    print(f"  [generate] Answer: {answer[:100]}...")
    return {"context": context, "answer": answer}


# Conditional edge function
def route_after_grading(state: RAGState) -> str:
    """Route to generate if chunks are relevant, rewrite if not"""
    if state["decision"] == "generate":
        return "generate"
    elif state["attempts"] >= 2:
        print(" [route] Max attempts reached - generating anyway")
        return "generate"
    else:
        return "rewrite"
    
# Build the graph
graph = StateGraph(RAGState)

graph.add_node("retrieve", retrieve)
graph.add_node("grade", grade_chunks)
graph.add_node("rewrite", rewrite_question)
graph.add_node("generate", generate_answer)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "grade")
graph.add_conditional_edges(
    "grade",
    route_after_grading,
    {
        "generate" : "generate",
        "rewrite" : "rewrite"
    }
)

graph.add_edge("rewrite", "retrieve")
graph.add_edge("generate", END)

corrective_rag = graph.compile()

questions = [
    "What is data engineering?",
    "xyzabc123 quantum blockchain metaverse",   # gibberish — should trigger rewrite
]

for question in questions:
    print(f"\nQuestion: {question}")
    print("─" * 60)
    result = corrective_rag.invoke({
        "question":           question,
        "retrieved_chunks":   [],
        "distances":          [],
        "relevance_scores":   [],
        "rewritten_question": "",
        "context":            "",
        "answer":             "",
        "attempts":           0,
        "decision":           "",
    })
    print(f"\nFinal answer: {result['answer'][:200]}...")
    if result["rewritten_question"]:
        print(f"Question was rewritten to: {result['rewritten_question']}")

d:\Projects\active\docmind\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4225.56it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded - 1416 chunks

Question: What is data engineering?
────────────────────────────────────────────────────────────
 [retrieve] Got 3 chunks for : 'What is data engineering?'
 [grade] relevant - Data engineering is a set of operations aimed at creating in...
 [grade] relevant - the shadows and now sharing the stage with data science. Dat...
 [grade] relevant - and processes that take in raw data and produce high -qualit...
 [grade] 3/3 relevant -> decision: generate
  [generate] Answer: According to the text, "Data engineering" is a set of operations aimed at creating interfaces and me...

Final answer: According to the text, "Data engineering" is a set of operations aimed at creating interfaces and mechanisms for the flow and access of information. It takes dedicated specialists—data engineers—to ma...

Question: xyzabc123 quantum blockchain metaverse
────────────────────────────────────────────────────────────
 [retrieve] Got 3 chunks for : 'xyzabc123 quantum blockchain metaverse'

In [2]:
# ── Stricter grader prompt ─────────────────────────────────────────────────
strict_grade_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a strict relevance grader for a RAG system.

Your job: decide if a document chunk contains information that DIRECTLY 
helps answer the question. Be strict — if the chunk is only loosely 
related or tangentially connected, mark it irrelevant.

Rules:
- 'relevant' ONLY if the chunk directly addresses the question
- 'irrelevant' if the chunk is on a different topic entirely
- 'irrelevant' if the question contains nonsense or gibberish

Respond with ONLY one word: 'relevant' or 'irrelevant'"""),
    ("human", """Question: {question}

Document chunk: {chunk}

Is this chunk directly relevant to answering the question?""")
])
strict_grade_chain = strict_grade_prompt | llm | parser

def grade_chunks_strict(state: RAGState) -> dict:
    """Grade each retrieved chunk with stricter relevance criteria."""
    question = state.get("rewritten_question") or state["question"]
    scores   = []
    for chunk in state["retrieved_chunks"]:
        score = strict_grade_chain.invoke({
            "question": question,
            "chunk":    chunk[:300]
        }).strip().lower()
        score = "relevant" if "relevant" in score and "irrelevant" not in score else "irrelevant"
        scores.append(score)
        print(f"  [grade] {score} — {chunk[:60]}...")

    relevant_count = scores.count("relevant")
    decision       = "generate" if relevant_count >= 1 else "rewrite"
    print(f"  [grade] {relevant_count}/3 relevant → decision: {decision}")
    return {"relevance_scores": scores, "decision": decision}

# ── Rebuild graph with strict grader ──────────────────────────────────────
graph2 = StateGraph(RAGState)

graph2.add_node("retrieve",  retrieve)
graph2.add_node("grade",     grade_chunks_strict)
graph2.add_node("rewrite",   rewrite_question)
graph2.add_node("generate",  generate_answer)

graph2.set_entry_point("retrieve")
graph2.add_edge("retrieve", "grade")
graph2.add_conditional_edges(
    "grade",
    route_after_grading,
    {
        "generate": "generate",
        "rewrite":  "rewrite"
    }
)
graph2.add_edge("rewrite",  "retrieve")
graph2.add_edge("generate", END)

corrective_rag_v2 = graph2.compile()

# ── Retest both questions ──────────────────────────────────────────────────
questions = [
    "What are the principles of good data architecture?",
    "xyzabc123 quantum blockchain metaverse",
]

for question in questions:
    print(f"\nQuestion: {question}")
    print("─" * 60)
    result = corrective_rag_v2.invoke({
        "question":           question,
        "retrieved_chunks":   [],
        "distances":          [],
        "relevance_scores":   [],
        "rewritten_question": "",
        "context":            "",
        "answer":             "",
        "attempts":           0,
        "decision":           "",
    })
    print(f"\nFinal answer: {result['answer'][:200]}...")
    if result["rewritten_question"]:
        print(f"Rewritten to: {result['rewritten_question']}")


Question: What are the principles of good data architecture?
────────────────────────────────────────────────────────────
 [retrieve] Got 3 chunks for : 'What are the principles of good data architecture?'
  [grade] relevant — Data Architecture Defined                                   ...
  [grade] relevant — aim to make significant decisions that will lead to good arc...
  [grade] relevant — Now that we have a working definition of data architecture, ...
  [grade] 3/3 relevant → decision: generate
  [generate] Answer: I don't find this in the document. The text provided only covers the definition of "good" data archi...

Final answer: I don't find this in the document. The text provided only covers the definition of "good" data architecture and mentions that it will be covered later. However, it does mention a principle related to ...

Question: xyzabc123 quantum blockchain metaverse
────────────────────────────────────────────────────────────
 [retrieve] Got 3 chunks for : 'xyzabc1

In [3]:
# ── Fix 1: Higher K for retrieval ─────────────────────────────────────────
def retrieve_k5(state: RAGState) -> dict:
    """Retrieve top-5 chunks — wider net."""
    question  = state.get("rewritten_question") or state["question"]
    query_emb = rag.embedding_model.encode([question]).astype(np.float32)
    dists, idxs = rag.index.search(query_emb, 5)
    chunks    = [rag.chunks[i] for i in idxs[0]]
    distances = [round(float(d), 4) for d in dists[0]]
    print(f"  [retrieve] Got {len(chunks)} chunks for: '{question[:60]}'")
    return {
        "retrieved_chunks": chunks,
        "distances":        distances,
        "relevance_scores": [],
    }

# ── Fix 2: Stricter routing — require 2+ relevant chunks ──────────────────
def route_after_grading_v2(state: RAGState) -> str:
    """
    Stricter routing:
    - 2+ relevant chunks → generate
    - 0-1 relevant chunks → rewrite (unless max attempts reached)
    """
    relevant_count = state["relevance_scores"].count("relevant")
    if relevant_count >= 2:
        return "generate"
    elif state["attempts"] >= 2:
        print("  [route] Max attempts reached — generating anyway")
        return "generate"
    else:
        return "rewrite"

# ── Rebuild with both fixes ────────────────────────────────────────────────
graph3 = StateGraph(RAGState)

graph3.add_node("retrieve",  retrieve_k5)
graph3.add_node("grade",     grade_chunks_strict)
graph3.add_node("rewrite",   rewrite_question)
graph3.add_node("generate",  generate_answer)

graph3.set_entry_point("retrieve")
graph3.add_edge("retrieve", "grade")
graph3.add_conditional_edges(
    "grade",
    route_after_grading_v2,
    {
        "generate": "generate",
        "rewrite":  "rewrite"
    }
)
graph3.add_edge("rewrite",  "retrieve")
graph3.add_edge("generate", END)

corrective_rag_v3 = graph3.compile()

# ── Retest ─────────────────────────────────────────────────────────────────
questions = [
    "What are the principles of good data architecture?",
    "xyzabc123 quantum blockchain metaverse",
    "What is the difference between batch and streaming data?",
]

for question in questions:
    print(f"\nQuestion: {question}")
    print("─" * 60)
    result = corrective_rag_v3.invoke({
        "question":           question,
        "retrieved_chunks":   [],
        "distances":          [],
        "relevance_scores":   [],
        "rewritten_question": "",
        "context":            "",
        "answer":             "",
        "attempts":           0,
        "decision":           "",
    })
    print(f"\nFinal answer: {result['answer'][:200]}...")
    if result["rewritten_question"]:
        print(f"Rewritten to: {result['rewritten_question']}")
    print()


Question: What are the principles of good data architecture?
────────────────────────────────────────────────────────────
  [retrieve] Got 5 chunks for: 'What are the principles of good data architecture?'
  [grade] relevant — Data Architecture Defined                                   ...
  [grade] relevant — aim to make significant decisions that will lead to good arc...
  [grade] relevant — Now that we have a working definition of data architecture, ...
  [grade] relevant — Additional Resources                                        ...
  [grade] relevant — Data Architecture 
A data architecture reflects the current ...
  [grade] 5/3 relevant → decision: generate
  [generate] Answer: According to the provided context, Principle 1 is:

Principle 1: Choose Common Components Wisely...

Final answer: According to the provided context, Principle 1 is:

Principle 1: Choose Common Components Wisely...


Question: xyzabc123 quantum blockchain metaverse
─────────────────────────────────────